# TalentSift build challenge

**MSBA Professional Development Workshop Series: Responsible AI System Design**

You have **20 minutes**. Read [PRD.md](../PRD.md) first, then:

1. Decide whether each requirement needs **ML, an agent, or neither** (fill in the table below).
2. Build R1, R2 and R3 by replacing the three functions marked `TODO`.
3. Run the last cell to write `outputs/results.csv`.

Any AI tool is allowed. If you use an API, load the key from an environment variable.

## Your decisions

Fill this in before you build, and update it if you change your mind.

| Requirement | ML, agent, or neither? | Why | Who checks the output? |
|---|---|---|---|
| R1 Extract | | | |
| R2 Filter | | | |
| R3 Rank | | | |

In [ ]:
from pathlib import Path
import pandas as pd

# Works whether you start Jupyter from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA, OUTPUTS = ROOT / "data", ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

## Load the inputs

In [ ]:
job_description = (DATA / "job_description.md").read_text()
applications = pd.read_csv(DATA / "applications.csv")

print(f"{len(applications)} applications")
applications.head()

## Optional: use Lumen

Lumen is the university-hosted LLM service (see the README for setup). Run these two cells once and you can call `ask_lumen("...")` from any of your R1 to R3 functions. The key is read from your environment, so never paste it into the notebook.

In [ ]:
import os

try:  # Colab: Secrets panel (key icon), name it LUMEN_API_KEY and allow notebook access
    from google.colab import userdata
    os.environ.setdefault("LUMEN_API_KEY", userdata.get("LUMEN_API_KEY"))
except ImportError:  # Local: .env file in the repo root
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")

print("Lumen key loaded" if os.environ.get("LUMEN_API_KEY") else "No LUMEN_API_KEY found (fine if you are not using Lumen)")

In [ ]:
from openai import OpenAI

# Lumen's chat-completions endpoint (the same code Lumen shows under "Use this model")
lumen = OpenAI(
    base_url="https://lumen.ncsa.illinois.edu/v1",
    api_key=os.environ.get("LUMEN_API_KEY", "not-set"),
)


def ask_lumen(prompt: str, model: str = "gemma-4-31b-it") -> str:
    resp = lumen.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


# print(ask_lumen("Hello!"))  # uncomment to test your key

### A basic text reader

This reads the text layer of a PDF and the paragraphs of a DOCX. It is a starting point, not a solution: it may miss content. Check what it returns before you trust it.

In [ ]:
import pdfplumber
import docx


def read_raw_text(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        with pdfplumber.open(path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    if path.suffix.lower() == ".docx":
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    raise ValueError(f"Unsupported file type: {path.name}")


sample = DATA / applications.loc[0, "resume_file"]
print(read_raw_text(sample)[:800])

## R1: Extract

Return one structured record per resume. Keep the keys below so the rest of the notebook works.

In [ ]:
def extract_resume(path: Path) -> dict:
    """TODO: replace with your R1 approach."""
    text = read_raw_text(path)
    return {
        "name": None,
        "email": None,
        "phone": None,
        "location": None,
        "education": [],   # list of {"degree", "school", "year"}
        "experience": [],  # list of {"title", "company", "start", "end"}
        "skills": [],
        "raw_text": text,
        "parse_error": None if text.strip() else "No text extracted",
    }


extract_resume(sample)

## R2: Filter

Decide whether an applicant meets the hard requirements in the job description. You get the applicant's row from `applications.csv` and their extracted record.

In [ ]:
def passes_hard_requirements(application: pd.Series, record: dict) -> tuple[bool, str]:
    """TODO: replace with your R2 approach. Return (passed, reason). Reason is blank if passed."""
    return True, ""

## R3: Rank

Score how well an applicant fits the job description. Higher is better. Return the score and the main reasons for it.

In [ ]:
def score_fit(record: dict, job_description: str) -> tuple[float, str]:
    """TODO: replace with your R3 approach. Return (score, reasons)."""
    return 0.0, ""

## Run the pipeline

In [ ]:
import time

rows = []
start = time.perf_counter()
for _, app in applications.iterrows():
    record = extract_resume(DATA / app["resume_file"])
    passed, reason = passes_hard_requirements(app, record)
    score, why = score_fit(record, job_description) if passed else (None, "")
    rows.append({
        "candidate_id": app["candidate_id"],
        "name": record.get("name"),
        "parse_error": record.get("parse_error"),
        "passed_filter": passed,
        "filter_reason": reason,
        "score": score,
        "reasons": why,
    })

elapsed = time.perf_counter() - start

results = pd.DataFrame(rows)
ranked = results[results["passed_filter"]].sort_values("score", ascending=False)
results.loc[ranked.index, "rank"] = range(1, len(ranked) + 1)
results["shortlisted"] = results["rank"] <= 10

results.to_csv(OUTPUTS / "results.csv", index=False)
print(f"Passed filter: {results['passed_filter'].sum()} / {len(results)}")
print(f"Could not read: {results['parse_error'].notna().sum()}")
print(f"Run time: {elapsed:.1f} s total, {elapsed / len(results):.2f} s per application")
results.sort_values("rank").head(10)

## Before the debrief

- What does it cost to run, in time and money per application? (Run time is printed above.)
- Would you trust it with a real person's job?
- What happens at 10 times the volume?
- Who checks its output?